In [ ]:
# ============================================
# TITANIC SURVIVAL PREDICTION
# ============================================

# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report,
                             roc_curve, auc)
import joblib
import warnings
warnings.filterwarnings('ignore')

# ตั้งค่าให้แสดงผลภาษาไทย (ถ้าต้องการ)
plt.rcParams['font.sans-serif'] = ['Angsana New', 'Leelawadee', 'DejaVu Sans']

print("="*60)
print("TITANIC SURVIVAL PREDICTION - MACHINE LEARNING PROJECT")
print("="*60)

# ============================================
# 1. LOAD DATASET
# ============================================
print("\n[1] โหลดข้อมูล...")

# โหลดข้อมูล (ถ้ามีไฟล์ train.csv)
try:
    df = pd.read_csv('data/train.csv')
    print(f"✓ โหลดข้อมูลสำเร็จ: {df.shape[0]} rows, {df.shape[1]} columns")
except:
    # ถ้าไม่มีไฟล์ ให้สร้างข้อมูลตัวอย่าง
    from sklearn.datasets import make_classification
    print("⚠ ไม่พบไฟล์ train.csv ใช้ข้อมูลตัวอย่างแทน")
    # ใช้ dataset จาก seaborn แทน
    df = sns.load_dataset('titanic')

print(f"\nขนาด Dataset: {df.shape}")
print(f"จำนวนผู้โดยสาร: {len(df)} คน")

# ============================================
# 2. DATA EXPLORATION
# ============================================
print("\n[2] สำรวจข้อมูล...")

print("\n📊 ข้อมูล 5 แถวแรก:")
print(df.head())

print("\n📋 ข้อมูลเบื้องต้น:")
print(df.info())

print("\n สถิติเชิงพรรณนา:")
print(df.describe())

print("\n❓ ตรวจสอบ Missing Values:")
print(df.isnull().sum())

# ============================================
# 3. DATA VISUALIZATION
# ============================================
print("\n[3] สร้างภาพ可视化...")

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 1. การกระจายของ Survived
axes[0, 0].pie(df['Survived'].value_counts(), 
               labels=['ไม่รอด', 'รอด'], 
               autopct='%1.1f%%',
               colors=['#ff6b6b', '#4ecdc4'])
axes[0, 0].set_title('อัตราการรอดชีวิต')

# 2. อายุ vs การรอดชีวิต
sns.boxplot(x='Survived', y='Age', data=df, ax=axes[0, 1])
axes[0, 1].set_title('อายุ vs การรอดชีวิต')

# 3. เพศ vs การรอดชีวิต
sns.countplot(x='Sex', hue='Survived', data=df, ax=axes[0, 2])
axes[0, 2].set_title('เพศ vs การรอดชีวิต')

# 4. ชั้นตั๋ว vs การรอดชีวิต
sns.countplot(x='Pclass', hue='Survived', data=df, ax=axes[1, 0])
axes[1, 0].set_title('ชั้นตั๋ว vs การรอดชีวิต')

# 5. Distribution ของอายุ
sns.histplot(df['Age'].dropna(), kde=True, ax=axes[1, 1])
axes[1, 1].set_title('การกระจายของอายุ')

# 6. ค่าตั๋ว vs การรอดชีวิต
sns.boxplot(x='Survived', y='Fare', data=df, ax=axes[1, 2])
axes[1, 2].set_title('ค่าตั๋ว vs การรอดชีวิต')

plt.tight_layout()
plt.savefig('exploratory_analysis.png', dpi=300)
print("✓ บันทึกกราฟ exploratory_analysis.png")
plt.show()

# ============================================
# 4. DATA PREPROCESSING
# ============================================
print("\n[4] Data Preprocessing...")

# ทำสำเนาข้อมูล
df_processed = df.copy()

print("\n ขั้นตอนการทำ Preprocessing:")

# 4.1 เลือก Features ที่ต้องการใช้
print("\n4.1 เลือก Features:")
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
df_processed = df_processed[features + ['Survived']]
print(f"   Features ที่ใช้: {features}")

# 4.2 จัดการ Missing Values
print("\n4.2 จัดการ Missing Values:")
print(f"   ก่อน: {df_processed.isnull().sum().sum()} missing values")

# เติม Age ด้วยค่าเฉลี่ย
age_imputer = SimpleImputer(strategy='mean')
df_processed['Age'] = age_imputer.fit_transform(df_processed[['Age']])

# เติม Embarked ด้วยค่าที่พบบ่อยที่สุด
embarked_imputer = SimpleImputer(strategy='most_frequent')
df_processed['Embarked'] = embarked_imputer.fit_transform(df_processed[['Embarked']])

print(f"   หลัง: {df_processed.isnull().sum().sum()} missing values")

# 4.3 Encoding ข้อมูล Categorical
print("\n4.3 Encoding ข้อมูล Categorical:")

# Sex: male=0, female=1
df_processed['Sex'] = df_processed['Sex'].map({'male': 0, 'female': 1})

# Embarked: One-Hot Encoding
df_processed = pd.get_dummies(df_processed, columns=['Embarked'], prefix='Embarked')

print("   ✓ Sex: male=0, female=1")
print("   ✓ Embarked: One-Hot Encoding")

# 4.4 แยก Features และ Target
print("\n4.4 แยก Features และ Target:")
X = df_processed.drop('Survived', axis=1)
y = df_processed['Survived']

print(f"   X (Features): {X.shape}")
print(f"   y (Target): {y.shape}")
print(f"   Features: {list(X.columns)}")

# 4.5 Split Data
print("\n4.5 Split Data (Train/Test):")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"   Training set: {X_train.shape[0]} samples")
print(f"   Testing set: {X_test.shape[0]} samples")

# 4.6 Feature Scaling
print("\n4.6 Feature Scaling (Standardization):")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("   ✓ ใช้ StandardScaler")

print("\n✅ Data Preprocessing เสร็จสมบูรณ์!")

# ============================================
# 5. MODEL TRAINING & COMPARISON
# ============================================
print("\n[5] Training Machine Learning Models...")

# กำหนด Models ที่จะเปรียบเทียบ
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'Naive Bayes': GaussianNB()
}

# ตารางเก็บผลลัพธ์
results = []

print("\n📊 Training และ Evaluation:")
print("-" * 80)

for name, model in models.items():
    print(f"\n🔹 {name}:")
    
    # Training
    model.fit(X_train_scaled, y_train)
    
    # Prediction
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, 'predict_proba') else None
    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    print(f"   Accuracy:  {accuracy:.4f}")
    print(f"   Precision: {precision:.4f}")
    print(f"   Recall:    {recall:.4f}")
    print(f"   F1-Score:  {f1:.4f}")
    
    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Model_Object': model,
        'Predictions': y_pred,
        'Probabilities': y_pred_proba
    })

# สร้าง DataFrame เปรียบเทียบ
results_df = pd.DataFrame(results)
print("\n" + "=" * 80)
print("📋 ตารางเปรียบเทียบ Models:")
print("=" * 80)
print(results_df[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score']].to_string(index=False))

# ============================================
# 6. VISUALIZATION OF RESULTS
# ============================================
print("\n[6] สร้างกราฟเปรียบเทียบ...")

# 6.1 Bar Chart เปรียบเทียบ Accuracy
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Accuracy Comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12']

for idx, metric in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    bars = ax.barh(results_df['Model'], results_df[metric], color=colors[idx])
    ax.set_xlabel(metric)
    ax.set_title(f'{metric} Comparison')
    ax.set_xlim([0, 1])
    
    # แสดงค่าบนแท่ง
    for i, (bar, val) in enumerate(zip(bars, results_df[metric])):
        ax.text(val + 0.02, bar.get_y() + bar.get_height()/2, 
                f'{val:.3f}', va='center', fontsize=9)

# Confusion Matrix ของ Model ที่ดีที่สุด
best_model_idx = results_df['Accuracy'].idxmax()
best_model_name = results_df.iloc[best_model_idx]['Model']
best_predictions = results_df.iloc[best_model_idx]['Predictions']

cm = confusion_matrix(y_test, best_predictions)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 2], 
            xticklabels=['Not Survived', 'Survived'],
            yticklabels=['Not Survived', 'Survived'])
axes[1, 2].set_title(f'Confusion Matrix - {best_model_name}')
axes[1, 2].set_xlabel('Predicted')
axes[1, 2].set_ylabel('Actual')

# ROC Curve
ax_roc = axes[1, 1]
for result in results:
    if result['Probabilities'] is not None:
        fpr, tpr, _ = roc_curve(y_test, result['Probabilities'])
        roc_auc = auc(fpr, tpr)
        ax_roc.plot(fpr, tpr, label=f"{result['Model']} (AUC = {roc_auc:.3f})")

ax_roc.plot([0, 1], [0, 1], 'k--', label='Random')
ax_roc.set_xlabel('False Positive Rate')
ax_roc.set_ylabel('True Positive Rate')
ax_roc.set_title('ROC Curve Comparison')
ax_roc.legend(loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300)
print("✓ บันทึกกราฟ model_comparison.png")
plt.show()

# ============================================
# 7. SAVE MODELS
# ============================================
print("\n[7] บันทึก Models...")

# บันทึก Model ที่ดีที่สุด
best_model = results_df.iloc[best_model_idx]['Model_Object']
joblib.dump(best_model, 'best_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print(f"✓ บันทึก best_model.pkl ({best_model_name})")
print(f"✓ บันทึก scaler.pkl")

# บันทึก preprocessing objects
joblib.dump({
    'age_imputer': age_imputer,
    'embarked_imputer': embarked_imputer,
    'feature_names': list(X.columns)
}, 'preprocessors.pkl')

print("✓ บันทึก preprocessors.pkl")

# ============================================
# 8. FINAL SUMMARY
# ============================================
print("\n" + "=" * 80)
print("📊 สรุปผลการทดลอง")
print("=" * 80)
print(f"\n✅ Model ที่ดีที่สุด: {best_model_name}")
print(f"   Accuracy: {results_df.iloc[best_model_idx]['Accuracy']:.4f}")
print(f"   Precision: {results_df.iloc[best_model_idx]['Precision']:.4f}")
print(f"   Recall: {results_df.iloc[best_model_idx]['Recall']:.4f}")
print(f"   F1-Score: {results_df.iloc[best_model_idx]['F1-Score']:.4f}")

print("\n📁 ไฟล์ที่สร้าง:")
print("   - exploratory_analysis.png")
print("   - model_comparison.png")
print("   - best_model.pkl")
print("   - scaler.pkl")
print("   - preprocessors.pkl")

print("\n🎉 Training เสร็จสมบูรณ์!")
print("=" * 80)